# Phase 5 LSI: LoRA, LoRMA, full fine-tuning and distillation on my phase 1 gqa_swa model

Attach the phase 1 output (`gqa_swa_best.pt` and `bpe_legal_16k_v2.pkl`) and add the `HF` secret. If you attach an earlier output dataset, the experiments that already have a zip and a checkpoint are skipped.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, re, json, math, time, shutil, pickle, random, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt
from huggingface_hub import login, hf_hub_download
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score
from IPython.display import FileLink, display


# timestamped and flushed, so the kaggle console updates while it trains
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for secret_name in ("HF_TOKEN", "HF"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(secret_name)
            if HF_TOKEN:
                break
        except Exception:
            pass
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("WARNING: HF_TOKEN not found, public datasets will download without auth.")

## Config

In [ ]:
# SMOKE = tiny dry run, SKIP_IF_ZIP_EXISTS = do not retrain an experiment whose zip and checkpoint were restored
SMOKE = False
RUN_PREFLIGHT = True
SKIP_IF_ZIP_EXISTS = True
LOG_EVERY = 100
SEED = 42

# every case is cut into overlapping chunks of CHUNK_LEN tokens, STRIDE tokens of overlap
CHUNK_LEN, STRIDE = 512, 128
MAX_CHUNKS_TRAIN, MAX_CHUNKS_EVAL = 3, 4
BATCH_SIZE = 16
# 2 batches per optimizer step, so 32 cases
GRAD_ACCUM = 2
WEIGHT_DECAY = 0.01

RANK, ALPHA = 8, 16.0
# the distilled student keeps every second layer of the phase 1 model
STUDENT_LAYERS = 8
KD_TEMP = 2.0
KD_WEIGHT = 0.4
FULL_LR = 2e-5
EXPERIMENTS = [
    {"num": 1, "name": "lora", "method": "lora", "backbone_lr": 3e-4, "head_lr": 5e-4},
    {"num": 2, "name": "lorma", "method": "lorma", "backbone_lr": 3e-4, "head_lr": 5e-4},
    {"num": 3, "name": "full", "method": "full", "backbone_lr": FULL_LR, "head_lr": 5e-4},
    {"num": 4, "name": "distillation", "method": "distillation", "backbone_lr": 2.5e-4, "head_lr": 5e-4, "kd": True},
]

# None = look for them in /kaggle/input
CKPT_PATH = None
TOK_PATH = None

# not stored in the checkpoint, same as phase 1
MAX_SEQ_LEN = 1023
ATTN_WINDOW, GLOBAL_EVERY = 256, 4

WORK_DIR = Path("/kaggle/working")
BASE_DIR = WORK_DIR / "phase5_lsi"
BASE_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
log(f"device: {DEVICE} | {torch.cuda.get_device_name(0) if USE_AMP else ''} | SMOKE={SMOKE}")

## Resume from an earlier session

In [ ]:
# finished zips and best checkpoints from an attached output dataset get copied back, so those experiments can be skipped
restored_zips, restored_ckpts = 0, 0
for root, _, files in os.walk("/kaggle/input"):
    for fname in files:
        src = os.path.join(root, fname)
        if fname.startswith("submission_exp") and fname.endswith(".zip"):
            dst = WORK_DIR / fname
            if not dst.exists():
                shutil.copy2(src, dst)
                print(f"restored zip: {dst} (from {src})")
                restored_zips += 1
        elif (fname.startswith("best_lsi_exp") or fname.startswith("resume_lsi_exp")) and fname.endswith(".pt"):
            dst = BASE_DIR / fname
            if not dst.exists():
                shutil.copy2(src, dst)
                print(f"restored checkpoint: {dst} (from {src})")
                restored_ckpts += 1

print(f"restored {restored_zips} zip(s) and {restored_ckpts} checkpoint(s)")

## Phase 1 checkpoint and tokenizer

In [ ]:
CANDIDATE_CKPT_PATHS = [
    "../input/nb2-checkpoint-8500/cs781_phase1/checkpoints_v3/gqa_swa_best.pt",
    "/kaggle/input/nb2_checkpoint_8500/cs781_phase1/checkpoints_v3/gqa_swa_best.pt",
    "/kaggle/input/nb2-checkpoint-8500/cs781_phase1/checkpoints_v3/gqa_swa_best.pt",
    "/kaggle/input/pretrained_phase2_himanshu/cs781_phase1/checkpoints_v3/gqa_swa_best.pt",
    "/kaggle/input/pretrained-phase2-himanshu/cs781_phase1/checkpoints_v3/gqa_swa_best.pt",
]
CANDIDATE_TOK_PATHS = [
    "../input/nb2-checkpoint-8500/cs781_phase1/tokenizer/bpe_legal_16k_v2.pkl",
    "/kaggle/input/nb2_checkpoint_8500/cs781_phase1/tokenizer/bpe_legal_16k_v2.pkl",
    "/kaggle/input/nb2-checkpoint-8500/cs781_phase1/tokenizer/bpe_legal_16k_v2.pkl",
    "/kaggle/input/pretrained_phase2_himanshu/cs781_phase1/tokenizer/bpe_legal_16k_v2.pkl",
    "/kaggle/input/pretrained-phase2-himanshu/cs781_phase1/tokenizer/bpe_legal_16k_v2.pkl",
]


# try the known paths first, then search /kaggle/input for the file name
def resolve_artifact(explicit_list, filename):
    for p in explicit_list:
        if os.path.isfile(p):
            return p
    hits = []
    for root, _, files in os.walk("/kaggle/input"):
        if filename in files:
            hits.append(os.path.join(root, filename))
    assert hits, f"{filename} not found in /kaggle/input, attach the phase 1 checkpoint dataset"
    return sorted(hits)[0]


if CKPT_PATH is None:
    CKPT_PATH = resolve_artifact(CANDIDATE_CKPT_PATHS, "gqa_swa_best.pt")
if TOK_PATH is None:
    TOK_PATH = resolve_artifact(CANDIDATE_TOK_PATHS, "bpe_legal_16k_v2.pkl")
print("checkpoint:", CKPT_PATH)
print("tokenizer :", TOK_PATH)


class CustomBPETokenizer:
    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\S|\s+(?!\S)|\s+""")
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = {}
        self._word_cache = {}

    def _tokenize_word(self, word):
        word_tuple = tuple(word)
        if len(word_tuple) <= 1:
            return list(word_tuple)
        while True:
            pairs = [(word_tuple[i], word_tuple[i + 1]) for i in range(len(word_tuple) - 1)]
            valid_merges = [p for p in pairs if p in self.merges]
            if not valid_merges:
                break
            best_pair = min(valid_merges, key=lambda p: self.vocab[self.merges[p]])
            first, second = best_pair
            merged_tok = self.merges[best_pair]
            new_tuple = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                    new_tuple.append(merged_tok)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_tuple)
            if len(word_tuple) == 1:
                break
        return list(word_tuple)

    def encode(self, text, add_special_tokens=False):
        if not text:
            return []
        cache = self._word_cache
        ids = []
        for word in self.pat.findall(text):
            word_ids = cache.get(word)
            if word_ids is None:
                word_ids = [self.vocab.get(tok, self.unk_token_id) for tok in self._tokenize_word(word)]
                if len(cache) < 2_000_000:
                    cache[word] = word_ids
            ids.extend(word_ids)
        if add_special_tokens:
            ids = [self.bos_token_id] + ids + [self.eos_token_id]
        return ids

    def decode(self, token_ids, skip_special_tokens=True):
        tokens = []
        for tid in token_ids:
            if skip_special_tokens and tid in (self.pad_token_id, self.bos_token_id, self.eos_token_id):
                continue
            tokens.append(self.inverse_vocab.get(tid, "<unk>"))
        return "".join(tokens)

    @property
    def vocab_size(self):
        return len(self.vocab)

    @classmethod
    def load(cls, filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        tok = cls(vocab_size=data["target_vocab_size"])
        tok.vocab = data["vocab"]
        tok.inverse_vocab = data["inverse_vocab"]
        tok.merges = data["merges"]
        tok.special_tokens = data["special_tokens"]
        print(f"Loaded tokenizer from {filepath} (Vocab size: {tok.vocab_size})")
        return tok


tokenizer = CustomBPETokenizer.load(TOK_PATH)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID, UNK_ID, BOS_ID, EOS_ID = tokenizer.pad_token_id, tokenizer.unk_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id

## Data

In [ ]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P5 = "Phase_5_LSI"


def fetch(rel):
    log(f"fetching {rel} from {HF_DATASET_REPO}")
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)


# labels can be a list, a single int or a string, we always want a sorted list of unique ints
def parse_statute_list(raw_labels):
    if raw_labels is None:
        return []
    if isinstance(raw_labels, (int, np.integer)):
        return [int(raw_labels)]
    if isinstance(raw_labels, str):
        raw_labels = [x.strip() for x in re.split(r"[,;\s]+", raw_labels) if x.strip()]
    out = set()
    for x in raw_labels:
        if isinstance(x, (int, np.integer)):
            out.add(int(x))
        elif isinstance(x, str):
            m = re.search(r"-?\d+", x)
            if m:
                out.add(int(m.group(0)))
    return sorted(out)


# handles a list of dicts, a dict of dicts or a dict of case_id -> text
def load_lsi_split(rel_path, is_test=False):
    local_path = fetch(rel_path)
    with open(local_path, "r", encoding="utf-8") as fh:
        raw = json.load(fh)

    if isinstance(raw, dict):
        if len(raw) == 1 and isinstance(next(iter(raw.values())), list) and next(iter(raw.keys())) in ("data", "examples", "cases", "items", "records"):
            raw_list = next(iter(raw.values()))
        elif "text" in raw and ("id" in raw or "doc_id" in raw or "case_id" in raw):
            raw_list = [raw]
        else:
            raw_list = []
            for k, v in raw.items():
                if isinstance(v, dict):
                    item = dict(v)
                    item.setdefault("id", str(k))
                    raw_list.append(item)
                elif isinstance(v, list):
                    raw_list.append({"id": str(k), "text": " ".join(str(x) for x in v)})
                elif isinstance(v, str):
                    raw_list.append({"id": str(k), "text": v})
    elif isinstance(raw, list):
        raw_list = raw
    else:
        raise ValueError(f"Unsupported JSON root type in {rel_path}: {type(raw)}")

    records = []
    for idx, item in enumerate(raw_list):
        if not isinstance(item, dict):
            continue
        cid = str(item.get("id", item.get("case_id", item.get("doc_id", item.get("uid", f"case_{idx}")))))
        text = item.get("text", item.get("document", item.get("judgment", item.get("facts", item.get("sentences", "")))))
        if isinstance(text, list):
            text = " ".join(str(s) for s in text)
        text = str(text).strip()

        labels = []
        if not is_test:
            for lk in ("labels", "label", "statutes", "statute_ids", "targets", "citations", "pred", "y"):
                if lk in item and item[lk] is not None:
                    labels = parse_statute_list(item[lk])
                    break

        records.append({"id": cid, "text": text, "labels": labels})

    assert len(records) > 0, f"loaded 0 records from {rel_path}"
    return records


train_data = load_lsi_split(f"{P5}/Train/train.json", is_test=False)
dev_data = load_lsi_split(f"{P5}/Train/dev.json", is_test=False)
test_data = load_lsi_split(f"{P5}/Test/test.json", is_test=True)

# statute id <-> column index
statute_counter = Counter()
for rec in train_data:
    statute_counter.update(rec["labels"])
for rec in dev_data:
    for s in rec["labels"]:
        statute_counter.setdefault(s, 0)

ALL_STATUTES = sorted(statute_counter.keys())
NUM_STATUTES = len(ALL_STATUTES)
assert NUM_STATUTES > 0, "no statute labels found in train.json or dev.json"

statute2idx = {stat_id: idx for idx, stat_id in enumerate(ALL_STATUTES)}
idx2statute = {idx: int(stat_id) for idx, stat_id in enumerate(ALL_STATUTES)}

# positives per statute in train
pos_counts = np.zeros(NUM_STATUTES, dtype=np.float64)
for rec in train_data:
    for s in rec["labels"]:
        if s in statute2idx:
            pos_counts[statute2idx[s]] += 1.0

N_train = float(len(train_data))
neg_counts = np.maximum(1.0, N_train - pos_counts)
# damped neg/pos ratio clipped to [1, 15], rare statutes get a boost without blowing up the logits
pos_weight_np = np.clip(np.sqrt(neg_counts / np.maximum(pos_counts, 1.0)), 1.0, 15.0)
POS_WEIGHT = torch.tensor(pos_weight_np, dtype=torch.float32, device=DEVICE)

# classifier bias starts at log(p / (1 - p)) of the train frequency of each statute
PRIOR_PROBS = np.clip(pos_counts / max(N_train, 1.0), 1e-4, 0.99)
INIT_BIAS_LOGITS = torch.tensor(np.log(PRIOR_PROBS / (1.0 - PRIOR_PROBS)), dtype=torch.float32)

train_label_lens = [len(x["labels"]) for x in train_data]
dev_label_lens = [len(x["labels"]) for x in dev_data]

print(f"train {len(train_data)} cases | avg statutes/case {np.mean(train_label_lens):.2f} (min {min(train_label_lens)}, max {max(train_label_lens)})")
print(f"dev   {len(dev_data)} cases | avg statutes/case {np.mean(dev_label_lens):.2f}")
print(f"test  {len(test_data)} cases | sample ids {[x['id'] for x in test_data[:3]]}")
print(f"{NUM_STATUTES} statutes | id range [{min(ALL_STATUTES)} .. {max(ALL_STATUTES)}]")
print("top 5 statutes (id, count):", statute_counter.most_common(5))
print("bottom 5 statutes (id, count):", statute_counter.most_common()[-5:])
print(f"pos_weight min {pos_weight_np.min():.2f} | median {np.median(pos_weight_np):.2f} | max {pos_weight_np.max():.2f}")

## Chunks and datasets

In [ ]:
# cut a case into overlapping windows of CHUNK_LEN tokens (stride STRIDE)
# too many windows -> keep evenly spaced ones, so we still see the start (facts), the middle and the end (final order)
def fast_encode_and_chunk_lsi(text, chunk_len=CHUNK_LEN, stride=STRIDE, max_chunks=MAX_CHUNKS_TRAIN):
    words = text.split()
    max_words_needed = max_chunks * 500
    if len(words) > max_words_needed * 2:
        # very long case: take head, middle and tail words before the bpe, so we do not tokenize 200k words
        seg_w = max_words_needed // 2
        mid_start = max(0, len(words) // 2 - seg_w // 2)
        sampled_words = words[:seg_w] + words[mid_start : mid_start + seg_w] + words[-seg_w:]
        token_ids = tokenizer.encode(" ".join(sampled_words), add_special_tokens=False)
    else:
        token_ids = tokenizer.encode(text, add_special_tokens=False)

    # room for bos and eos
    eff_len = chunk_len - 2
    if not token_ids:
        token_ids = [UNK_ID]

    raw_windows = []
    n = len(token_ids)
    if n <= eff_len:
        raw_windows.append(token_ids)
    else:
        step = max(1, chunk_len - stride)
        for start in range(0, n, step):
            win = token_ids[start : start + eff_len]
            raw_windows.append(win)
            if start + eff_len >= n:
                break

    if max_chunks is not None and len(raw_windows) > max_chunks:
        # evenly spaced, first and last window always included
        idxs = np.linspace(0, len(raw_windows) - 1, num=max_chunks, dtype=int)
        raw_windows = [raw_windows[i] for i in idxs]

    chunks, lengths = [], []
    for win in raw_windows:
        seq = [BOS_ID] + win + [EOS_ID]
        L = len(seq)
        if L < chunk_len:
            seq = seq + [PAD_ID] * (chunk_len - L)
        chunks.append(seq)
        lengths.append(L)

    return np.asarray(chunks, dtype=np.int16), np.asarray(lengths, dtype=np.int16)


# tokenize everything once and keep it in ram
def build_lsi_cache(records, max_chunks, split_name):
    t0 = time.time()
    cached = []
    total = len(records)
    for i, rec in enumerate(records, 1):
        chunks, lengths = fast_encode_and_chunk_lsi(rec["text"], CHUNK_LEN, STRIDE, max_chunks)
        label_idxs = [statute2idx[s] for s in rec["labels"] if s in statute2idx]
        cached.append({
            "id": rec["id"],
            "chunks": chunks,
            "lengths": lengths,
            "label_idxs": np.asarray(label_idxs, dtype=np.int32),
        })
        if i % 5000 == 0 or i == total:
            log(f"  [{split_name}] tokenized and chunked {i}/{total} cases ({i / total:.0%}) in {time.time() - t0:.1f}s")
    return cached


if SMOKE:
    print("SMOKE: 128 train, 64 dev and the full test set")
    active_train_records = train_data[:128]
    active_dev_records = dev_data[:64]
else:
    active_train_records = train_data
    active_dev_records = dev_data

# big train set -> 2 epochs, otherwise 3 (SMOKE only does 1)
EPOCHS = 1 if SMOKE else (2 if len(active_train_records) > 20000 else 3)

print("building the chunk caches...")
TRAIN_CACHE = build_lsi_cache(active_train_records, MAX_CHUNKS_TRAIN, "TRAIN")
DEV_CACHE = build_lsi_cache(active_dev_records, MAX_CHUNKS_EVAL, "DEV")
TEST_CACHE = build_lsi_cache(test_data, MAX_CHUNKS_EVAL, "TEST")


class LSIDataset(Dataset):
    def __init__(self, cached_records, num_statutes=NUM_STATUTES):
        self.records = cached_records
        self.num_statutes = num_statutes

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        # multi-hot target over all statutes
        multi_hot = torch.zeros(self.num_statutes, dtype=torch.float32)
        if len(rec["label_idxs"]) > 0:
            multi_hot[rec["label_idxs"]] = 1.0
        return {
            "id": rec["id"],
            "chunks": torch.from_numpy(rec["chunks"].astype(np.int64)),
            "lengths": torch.from_numpy(rec["lengths"].astype(np.int64)),
            "labels": multi_hot,
        }


# all chunks of the batch go into one flat tensor, doc_counts remembers which chunks belong to which case
def lsi_collate_fn(batch):
    ids = [b["id"] for b in batch]
    chunk_tensors = [b["chunks"] for b in batch]
    len_tensors = [b["lengths"] for b in batch]
    doc_counts = [c.size(0) for c in chunk_tensors]

    flat_chunks = torch.cat(chunk_tensors, dim=0)
    flat_lengths = torch.cat(len_tensors, dim=0)
    labels = torch.stack([b["labels"] for b in batch], dim=0)

    return {
        "ids": ids,
        "flat_chunks": flat_chunks,
        "flat_lengths": flat_lengths,
        "doc_counts": doc_counts,
        "labels": labels,
    }


train_dataset = LSIDataset(TRAIN_CACHE)
dev_dataset = LSIDataset(DEV_CACHE)
test_dataset = LSIDataset(TEST_CACHE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lsi_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lsi_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lsi_collate_fn)

_sample_b = next(iter(train_loader))
print(f"train batches/epoch {len(train_loader)} (EPOCHS={EPOCHS}) | dev batches {len(dev_loader)} | test batches {len(test_loader)}")
print(f"sample batch: {len(_sample_b['ids'])} cases, flat chunks {tuple(_sample_b['flat_chunks'].shape)}, targets {tuple(_sample_b['labels'].shape)}")

## Phase 1 model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(variance + self.eps))


class SwiGLU(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.w1 = nn.Linear(in_features, hidden_features, bias=False)
        self.w2 = nn.Linear(in_features, hidden_features, bias=False)
        self.w3 = nn.Linear(hidden_features, in_features, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


_ROPE = {}


def _rope_tables(head_dim, device, max_len=2048):
    key = (head_dim, str(device))
    if key not in _ROPE:
        # rotary position tables
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
        t = torch.arange(max_len, device=device).float()
        emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
        _ROPE[key] = (emb.cos(), emb.sin())
    return _ROPE[key]


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k):
    T, hd = q.size(-2), q.size(-1)
    cos, sin = _rope_tables(hd, q.device)
    cos = cos[:T].to(q.dtype)[None, None]
    sin = sin[:T].to(q.dtype)[None, None]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


_MASK = {}


def causal_window_mask(T, window, device):
    key = (T, window, str(device))
    m = _MASK.get(key)
    if m is None:
        _MASK.clear()
        i = torch.arange(T, device=device)[:, None]
        j = torch.arange(T, device=device)[None, :]
        # look back only, and only `window` tokens
        m = (j <= i) & ((i - j) < window)
        _MASK[key] = m
    return m


def attend(q, k, v, window=None):
    T = q.size(-2)
    # global layer or short input: plain causal attention
    if window is None or window >= T:
        return F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=causal_window_mask(T, window, q.device))


def init_weights(model, n_layers):
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    model.apply(_init)
    # smaller init for the output projections
    for name, p in model.named_parameters():
        if name.endswith("o_proj.weight") or name.endswith("w3.weight"):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * max(1, n_layers)))


# fewer k/v heads than q heads (grouped query)
class GQACausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window=None):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads
        self.window = window
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k)
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
        y = attend(q, k, v, self.window)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, window=None):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQACausalAttention(d_model, n_heads, n_kv_heads, window)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


class _CausalLMBase(nn.Module):
    def __init__(self, vocab_size, d_model, layers, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.use_grad_ckpt = True
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(layers)
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        init_weights(self, len(layers))

    def forward_features(self, idx):
        x = self.tok_emb(idx)
        for layer in self.layers:
            # checkpoint the blocks while training, so full ft fits on the gpu
            if self.use_grad_ckpt and self.training and x.requires_grad:
                x = grad_ckpt(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.final_norm(x)


class LegalEfficientTransformer(_CausalLMBase):
    def __init__(self, vocab_size, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816,
                 window=ATTN_WINDOW, global_every=GLOBAL_EVERY, max_seq_len=MAX_SEQ_LEN):
        layers = [
            EfficientDecoderBlock(
                d_model, n_heads, n_kv_heads, d_ff,
                window=None if (i + 1) % global_every == 0 else window,
            )
            for i in range(n_layers)
        ]
        super().__init__(vocab_size, d_model, layers, max_seq_len)


_ckpt_payload = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
_raw_sd = _ckpt_payload.get("state_dict", _ckpt_payload.get("model", _ckpt_payload))
BASE_STATE_DICT = {k.replace("_orig_mod.", "", 1): v for k, v in _raw_sd.items()}

# the checkpoint may carry its own config, otherwise read the sizes off the weights
if isinstance(_ckpt_payload, dict) and "config" in _ckpt_payload:
    BASE_CONFIG = dict(_ckpt_payload["config"])
else:
    _d_model = BASE_STATE_DICT["tok_emb.weight"].shape[1]
    _d_ff = BASE_STATE_DICT["layers.0.ffn.w1.weight"].shape[0]
    _n_layers = max(int(k.split(".")[1]) for k in BASE_STATE_DICT if k.startswith("layers.")) + 1
    _n_heads = 16 if _d_model == 1024 else 12
    BASE_CONFIG = dict(
        vocab_size=VOCAB_SIZE, d_model=_d_model, n_heads=_n_heads,
        n_kv_heads=4, n_layers=_n_layers, d_ff=_d_ff,
    )


# a fresh copy of phase 1 every time, so experiments never share weights
def load_fresh_backbone(device=DEVICE):
    backbone = LegalEfficientTransformer(**BASE_CONFIG)
    backbone.load_state_dict(BASE_STATE_DICT, strict=True)
    return backbone.to(device)


_test_bb = load_fresh_backbone("cpu")
print("strict checkpoint load ok:", BASE_CONFIG)
print(f"pretraining step {_ckpt_payload.get('step', 'N/A')} | val loss {_ckpt_payload.get('val_loss', 'N/A')}")
print(f"backbone parameters: {sum(p.numel() for p in _test_bb.parameters()) / 1e6:.2f}M")
del _test_bb, _ckpt_payload, _raw_sd
gc.collect()

## Classifier

In [ ]:
# frozen W plus a small trainable B @ A
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank, alpha, dropout=0.05):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)
        out_f, in_f = base_linear.weight.shape
        self.A = nn.Parameter(torch.empty(rank, in_f))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        # B starts at zero so at the start it behaves exactly like phase 1
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / float(rank)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # W x + scale * B A x
        return self.base(x) + self.scale * ((self.drop(x) @ self.A.t()) @ self.B.t())


# same idea but the update acts on the output of W instead of the input
class LoRMALinear(nn.Module):
    def __init__(self, base_linear, rank, alpha, dropout=0.05):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)
        out_f = base_linear.weight.size(0)
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / float(rank)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * ((self.drop(h) @ self.A.t()) @ self.B.t())


# every chunk goes through the backbone, the chunk vectors are pooled per case and a linear layer gives one logit per statute
class LegalStatuteModel(nn.Module):
    def __init__(self, backbone, num_statutes=NUM_STATUTES, chunk_micro_batch=32):
        super().__init__()
        self.backbone = backbone
        self.d_model = backbone.tok_emb.embedding_dim
        self.num_statutes = num_statutes
        self.chunk_micro_batch = chunk_micro_batch

        # mean + max + last token of a chunk -> one chunk vector
        self.chunk_proj = nn.Sequential(
            nn.Linear(3 * self.d_model, self.d_model),
            nn.LayerNorm(self.d_model),
            nn.GELU(),
            nn.Dropout(0.15),
        )

        # gated attention over the chunks of a case
        self.attn_V = nn.Linear(self.d_model, self.d_model // 2)
        self.attn_U = nn.Linear(self.d_model, self.d_model // 2)
        self.attn_w = nn.Linear(self.d_model // 2, 1, bias=False)

        # attention pooled + max pooled + mean pooled -> case vector
        self.doc_fusion = nn.Sequential(
            nn.Linear(3 * self.d_model, self.d_model),
            nn.LayerNorm(self.d_model),
            nn.GELU(),
        )
        # multi-sample dropout, the logits of the 5 dropout rates are averaged
        self.ms_dropouts = nn.ModuleList([nn.Dropout(p) for p in (0.1, 0.2, 0.3, 0.4, 0.5)])
        self.classifier = nn.Linear(self.d_model, num_statutes)

        nn.init.normal_(self.classifier.weight, std=0.02)
        with torch.no_grad():
            self.classifier.bias.copy_(INIT_BIAS_LOGITS)

    # run the chunks through the backbone a few at a time
    def _encode_chunks(self, flat_chunks, flat_lengths):
        total_c, T = flat_chunks.shape
        device = flat_chunks.device
        chunk_embs = []

        for s in range(0, total_c, self.chunk_micro_batch):
            sub_chunks = flat_chunks[s : s + self.chunk_micro_batch]
            sub_lens = flat_lengths[s : s + self.chunk_micro_batch]
            hs = self.backbone.forward_features(sub_chunks)

            # only the real tokens count, not the padding
            arange_t = torch.arange(T, device=device).unsqueeze(0)
            valid_mask = (arange_t < sub_lens.unsqueeze(1)).unsqueeze(-1)
            mask_f = valid_mask.to(hs.dtype)

            mean_p = (hs * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
            masked_hs = hs.masked_fill(~valid_mask, -1e4)
            max_p = masked_hs.max(dim=1).values
            last_idx = (sub_lens - 1).clamp(min=0)
            eos_p = hs[torch.arange(hs.size(0), device=device), last_idx]

            combined = torch.cat([mean_p, max_p, eos_p], dim=-1)
            chunk_embs.append(self.chunk_proj(combined))

        return torch.cat(chunk_embs, dim=0)

    def forward(self, flat_chunks, flat_lengths, doc_counts):
        chunk_vecs = self._encode_chunks(flat_chunks, flat_lengths)
        doc_splits = torch.split(chunk_vecs, doc_counts, dim=0)

        case_logits = []
        for doc_chunks in doc_splits:
            gates = torch.tanh(self.attn_V(doc_chunks)) * torch.sigmoid(self.attn_U(doc_chunks))
            scores = self.attn_w(gates)
            weights = F.softmax(scores, dim=0)

            attn_pooled = (weights * doc_chunks).sum(dim=0)
            max_pooled = doc_chunks.max(dim=0).values
            mean_pooled = doc_chunks.mean(dim=0)

            doc_vec = self.doc_fusion(torch.cat([attn_pooled, max_pooled, mean_pooled], dim=-1))
            logits = torch.stack([self.classifier(drop(doc_vec)) for drop in self.ms_dropouts], dim=0).mean(dim=0)
            case_logits.append(logits)

        # [batch, NUM_STATUTES]
        return torch.stack(case_logits, dim=0)


# freeze the phase 1 weights and wrap the target linears (all weights stay trainable for full)
def inject_adapters(backbone, method):
    for p in backbone.parameters():
        p.requires_grad_(False)

    if method == "full":
        for p in backbone.parameters():
            p.requires_grad_(True)

    elif method == "lora":
        # attention projections and the swiglu mlp
        wrapped = 0
        for blk in backbone.layers:
            for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(blk.attn, name, LoRALinear(getattr(blk.attn, name), RANK, ALPHA))
                wrapped += 1
            for name in ("w1", "w2", "w3"):
                setattr(blk.ffn, name, LoRALinear(getattr(blk.ffn, name), RANK, ALPHA))
                wrapped += 1
        for p in backbone.final_norm.parameters():
            p.requires_grad_(True)
        print(f"[LORA] wrapped {wrapped} linears (r={RANK}, alpha={ALPHA})")

    elif method == "lorma":
        # attention projections only
        wrapped = 0
        for blk in backbone.layers:
            for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(blk.attn, name, LoRMALinear(getattr(blk.attn, name), RANK, ALPHA))
                wrapped += 1
            for p in blk.attn_norm.parameters():
                p.requires_grad_(True)
        for p in backbone.final_norm.parameters():
            p.requires_grad_(True)
        print(f"[LORMA] wrapped {wrapped} linears (r={RANK}, alpha={ALPHA})")

    elif method == "distillation":
        # small student: every second phase 1 layer, lora on attention, top 2 layers trainable
        total_l = len(backbone.layers)
        even_indices = list(range(0, total_l, 2))[:STUDENT_LAYERS]
        backbone.layers = nn.ModuleList([backbone.layers[i] for i in even_indices])
        for blk in backbone.layers:
            for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(blk.attn, name, LoRALinear(getattr(blk.attn, name), RANK, ALPHA))
        for blk in backbone.layers[-2:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in backbone.final_norm.parameters():
            p.requires_grad_(True)
        print(f"[DISTILLATION] {len(backbone.layers)}-layer student built from phase 1 layers {even_indices}")

    else:
        raise ValueError(f"unknown method: {method}")

    return backbone


def make_model(method):
    backbone = inject_adapters(load_fresh_backbone(device="cpu"), method)
    model = LegalStatuteModel(backbone, num_statutes=NUM_STATUTES, chunk_micro_batch=32).to(DEVICE)

    tr_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot_params = sum(p.numel() for p in model.parameters())
    print(f"[{method.upper()}] trainable {tr_params / 1e6:.2f}M / total {tot_params / 1e6:.2f}M ({tr_params / tot_params:.1%})")
    return model

## Evaluation and submission

In [ ]:
# raw logits for a whole split, the model never sees the labels here
@torch.no_grad()
def get_split_logits(model, loader, desc=None):
    model.eval()
    all_ids, all_logits, all_labels = [], [], []
    total_b = len(loader)
    t0 = time.time()
    for b_idx, batch in enumerate(loader, 1):
        flat_chunks = batch["flat_chunks"].to(DEVICE, non_blocking=True)
        flat_lengths = batch["flat_lengths"].to(DEVICE, non_blocking=True)
        doc_counts = batch["doc_counts"]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(flat_chunks, flat_lengths, doc_counts)
        all_ids.extend(batch["ids"])
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(batch["labels"].numpy())
        if desc and (b_idx % 100 == 0 or b_idx == total_b):
            log(f"  [{desc}] {b_idx}/{total_b} batches ({b_idx / total_b:.0%}) in {time.time() - t0:.1f}s")
    return all_ids, np.concatenate(all_logits, axis=0), np.concatenate(all_labels, axis=0)


# thresholds can be one number or one per statute, a case with nothing predicted gets its top-1 statute
def apply_thresholds_with_top1_fallback(probs_np, thresholds):
    thr = np.asarray(thresholds)
    if thr.ndim == 0:
        thr = thr[None, None]
    elif thr.ndim == 1:
        thr = thr[None, :]
    preds = (probs_np >= thr).astype(np.int32)
    empty_rows = np.where(preds.sum(axis=1) == 0)[0]
    if len(empty_rows) > 0:
        top1_cols = probs_np[empty_rows].argmax(axis=1)
        preds[empty_rows, top1_cols] = 1
    return preds


# pick thresholds on dev to maximise macro f1: one global value first, then one per statute
# statutes with fewer than 2 dev positives keep the global value, so we do not overfit
def calibrate_thresholds(val_probs, val_labels):
    grid = np.linspace(0.15, 0.85, 29)
    best_global_tau = 0.50
    best_global_f1 = -1.0

    for tau in grid:
        preds = (val_probs >= tau).astype(np.int32)
        tp = (preds * val_labels).sum(axis=0)
        fp = (preds * (1 - val_labels)).sum(axis=0)
        fn = ((1 - preds) * val_labels).sum(axis=0)
        f1_per_class = (2.0 * tp) / np.maximum(2.0 * tp + fp + fn, 1e-8)
        macro_f1 = float(f1_per_class.mean())
        if macro_f1 > best_global_f1:
            best_global_f1 = macro_f1
            best_global_tau = float(tau)

    num_c = val_probs.shape[1]
    per_class_taus = np.full(num_c, best_global_tau, dtype=np.float32)
    pos_support = val_labels.sum(axis=0)

    for c in range(num_c):
        if pos_support[c] < 2:
            continue
        p_c = val_probs[:, c : c + 1]
        y_c = val_labels[:, c : c + 1]
        preds_grid = (p_c >= grid[None, :]).astype(np.float32)
        tp = (preds_grid * y_c).sum(axis=0)
        fp = (preds_grid * (1.0 - y_c)).sum(axis=0)
        fn = ((1.0 - preds_grid) * y_c).sum(axis=0)
        f1_grid = (2.0 * tp) / np.maximum(2.0 * tp + fp + fn, 1e-8)
        best_idx = int(f1_grid.argmax())
        if f1_grid[best_idx] > 0:
            per_class_taus[c] = float(grid[best_idx])

    # keep the per-statute thresholds only if they really beat the global one
    preds_global = apply_thresholds_with_top1_fallback(val_probs, best_global_tau)
    f1_global = f1_score(val_labels, preds_global, average="macro", zero_division=0)

    preds_per_cls = apply_thresholds_with_top1_fallback(val_probs, per_class_taus)
    f1_per_cls = f1_score(val_labels, preds_per_cls, average="macro", zero_division=0)

    if f1_per_cls >= f1_global:
        return per_class_taus, float(best_global_tau)
    return np.full(num_c, best_global_tau, dtype=np.float32), float(best_global_tau)


# macro f1 (main score), micro f1 and subset accuracy on the dev set
def evaluate_dev(model, dev_loader, thresholds=None, calibrate=True, verbose=False):
    _, logits, labels = get_split_logits(model, dev_loader, desc="Dev Eval" if verbose else None)
    logits_t = torch.from_numpy(logits).to(DEVICE)
    labels_t = torch.from_numpy(labels).to(DEVICE)
    val_loss = F.binary_cross_entropy_with_logits(logits_t, labels_t, pos_weight=POS_WEIGHT).item()

    probs = torch.sigmoid(logits_t).cpu().numpy()
    labels_int = labels.astype(np.int32)

    if calibrate or thresholds is None:
        thresholds, global_tau = calibrate_thresholds(probs, labels_int)
    else:
        global_tau = float(np.mean(thresholds))

    preds = apply_thresholds_with_top1_fallback(probs, thresholds)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(labels_int, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels_int, preds, average="micro", zero_division=0)
    subset_acc = accuracy_score(labels_int, preds)

    if verbose:
        avg_pred_cnt = preds.sum(axis=1).mean()
        avg_true_cnt = labels_int.sum(axis=1).mean()
        print(f"dev macro F1 {macro_f1:.4f} | micro F1 {micro_f1:.4f} | subset acc {subset_acc:.4f}")
        print(f"dev macro P {macro_p:.4f} R {macro_r:.4f} | mean threshold {np.mean(thresholds):.3f}")
        print(f"avg predicted statutes/case {avg_pred_cnt:.2f} (ground truth {avg_true_cnt:.2f})")

    return {
        "val_loss": float(val_loss),
        "macro_f1": float(macro_f1),
        "micro_f1": float(micro_f1),
        "subset_acc": float(subset_acc),
        "precision": float(macro_p),
        "recall": float(macro_r),
        "thresholds": thresholds,
        "global_tau": global_tau,
    }


# a list of statute ids for every test case, zipped as prediction.json
def generate_submission(model, thresholds, exp_num, method_name, loader=None, records=None):
    loader = test_loader if loader is None else loader
    records = test_data if records is None else records

    pred_ids, test_logits, _ = get_split_logits(model, loader, desc="Test Inference")
    test_probs = torch.sigmoid(torch.from_numpy(test_logits)).numpy()
    binary_preds = apply_thresholds_with_top1_fallback(test_probs, thresholds)

    valid_statute_set = set(ALL_STATUTES)
    id_to_pred_list = {}
    for cid, row_mask in zip(pred_ids, binary_preds):
        active_indices = np.where(row_mask == 1)[0].tolist()
        stat_ids = sorted({int(idx2statute[idx]) for idx in active_indices})
        assert len(stat_ids) > 0, f"empty prediction list for case {cid}"
        for s in stat_ids:
            assert isinstance(s, int) and s in valid_statute_set, f"invalid statute id {s} for case {cid}"
        id_to_pred_list[str(cid)] = stat_ids

    # same ids, same order as test.json
    expected_ids = [str(r["id"]) for r in records]
    submission_list = []
    for cid in expected_ids:
        assert cid in id_to_pred_list, f"missing prediction for test id {cid}"
        submission_list.append({"id": cid, "pred": id_to_pred_list[cid]})

    assert len(submission_list) == len(records), "prediction count does not match test.json"
    assert [x["id"] for x in submission_list] == expected_ids, "test id order mismatch"

    pred_path = BASE_DIR / f"prediction_{method_name}.json"
    zip_name = WORK_DIR / f"submission_exp{exp_num}_{method_name}.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(submission_list, f, indent=2, ensure_ascii=False)

    # prediction.json has to be at the root of the zip
    with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")

    # read the zip back to make sure it is fine
    with zipfile.ZipFile(zip_name, "r") as zf:
        assert zf.namelist() == ["prediction.json"], f"zip root error: {zf.namelist()}"
        ver = json.loads(zf.read("prediction.json").decode("utf-8"))
        assert len(ver) == len(records)

    avg_preds = np.mean([len(x["pred"]) for x in submission_list])
    log(f"wrote {zip_name} ({zip_name.stat().st_size / 1024:.1f} KB) | {len(submission_list)} cases, avg {avg_preds:.2f} statutes/case")
    print("sample:", submission_list[0])
    try:
        display(FileLink(str(zip_name.relative_to(WORK_DIR))))
    except Exception:
        pass
    return zip_name

## Sanity check

In [ ]:
# one tiny train step, a dev eval with calibration and a throwaway zip, so a broken setup fails now and not after an hour
if RUN_PREFLIGHT:
    log("running the pre-flight check...")
    _sm_tr = DataLoader(LSIDataset(TRAIN_CACHE[:16]), batch_size=4, shuffle=False, collate_fn=lsi_collate_fn)
    _sm_dv = DataLoader(LSIDataset(DEV_CACHE[:16]), batch_size=4, shuffle=False, collate_fn=lsi_collate_fn)
    _sm_ts = DataLoader(LSIDataset(TEST_CACHE[:16]), batch_size=4, shuffle=False, collate_fn=lsi_collate_fn)

    _sm_m = make_model("lorma")
    _sm_o = torch.optim.AdamW([p for p in _sm_m.parameters() if p.requires_grad], lr=1e-4)
    _sm_s = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    _sm_m.train()
    for _b in _sm_tr:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            _lg = _sm_m(_b["flat_chunks"].to(DEVICE), _b["flat_lengths"].to(DEVICE), _b["doc_counts"])
            _ls = F.binary_cross_entropy_with_logits(_lg.float(), _b["labels"].to(DEVICE), pos_weight=POS_WEIGHT)
        _sm_o.zero_grad(set_to_none=True)
        _sm_s.scale(_ls).backward()
        _sm_s.step(_sm_o)
        _sm_s.update()
        break

    _sm_met = evaluate_dev(_sm_m, _sm_dv, calibrate=True, verbose=False)
    _sm_zip = generate_submission(_sm_m, _sm_met["thresholds"], 0, "smoke", _sm_ts, test_data[:16])
    if _sm_zip.exists():
        _sm_zip.unlink()

    del _sm_m, _sm_o, _sm_s, _sm_tr, _sm_dv, _sm_ts
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()
    log("pre-flight check passed")

## Training loop

In [ ]:
# hard bce plus the temperature kl against the teacher (binary, on sigmoid probabilities)
def kd_loss_fn(student_logits, teacher_logits, targets, pos_weight=POS_WEIGHT, alpha=KD_WEIGHT, temp=KD_TEMP):
    s_lg = student_logits.float()
    t_lg = teacher_logits.float()
    y = targets.float()

    bce_loss = F.binary_cross_entropy_with_logits(s_lg, y, pos_weight=pos_weight)

    t_probs = torch.sigmoid(t_lg / temp).detach()
    kd_bce = F.binary_cross_entropy_with_logits(s_lg / temp, t_probs, reduction="none")
    # statutes the teacher finds likely count more than the easy zeros
    importance = 1.0 + (pos_weight.unsqueeze(0) - 1.0) * t_probs
    kd_loss = (kd_bce * importance).mean() * (temp ** 2)

    return alpha * bce_loss + (1.0 - alpha) * kd_loss


# only the weights that train are saved, the frozen phase 1 weights are loaded fresh again
def trainable_state_dict(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}


# trains for a fixed number of epochs, calibrates thresholds and prints dev scores every epoch
# teacher_map given = distillation, the best dev epoch is kept on disk in save_best_to
def run_training(model, exp, epochs, teacher_map=None, save_best_to=None):
    num, tag = exp["num"], exp["name"]
    backbone_lr, head_lr = exp["backbone_lr"], exp["head_lr"]

    # the new head has its own lr, 1d params (norms, biases) get no weight decay
    head_params, bb_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("backbone."):
            bb_params.append(p)
        else:
            head_params.append(p)

    param_groups = [{"params": head_params, "lr": head_lr, "weight_decay": WEIGHT_DECAY}]
    decay_p = [p for p in bb_params if p.ndim >= 2]
    nodecay_p = [p for p in bb_params if p.ndim < 2]
    if decay_p:
        param_groups.append({"params": decay_p, "lr": backbone_lr, "weight_decay": WEIGHT_DECAY})
    if nodecay_p:
        param_groups.append({"params": nodecay_p, "lr": backbone_lr, "weight_decay": 0.0})

    optimizer = torch.optim.AdamW(param_groups)
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    total_steps = max(1, steps_per_epoch * epochs)
    warmup_steps = max(1, int(0.10 * total_steps))

    # 10% warmup, then cosine down to 5% of the lr
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda s: float(s + 1) / warmup_steps if s < warmup_steps
        else max(0.05, 0.5 * (1.0 + math.cos(math.pi * float(s - warmup_steps) / max(1, total_steps - warmup_steps))))
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    trainable_params = head_params + bb_params

    best_f1 = -1.0
    exp_t0 = time.time()
    num_batches = len(train_loader)

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        ep_t0 = time.time()

        log(f"--> [{tag}] epoch {epoch}/{epochs} ({num_batches} batches)")
        for b_idx, batch in enumerate(train_loader, 1):
            flat_chunks = batch["flat_chunks"].to(DEVICE, non_blocking=True)
            flat_lengths = batch["flat_lengths"].to(DEVICE, non_blocking=True)
            doc_counts = batch["doc_counts"]
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(flat_chunks, flat_lengths, doc_counts)

                if teacher_map is not None:
                    t_logits = torch.tensor(
                        np.stack([teacher_map[cid] for cid in batch["ids"]], axis=0),
                        dtype=torch.float32, device=DEVICE
                    )
                    loss = kd_loss_fn(logits, t_logits, labels)
                else:
                    loss = F.binary_cross_entropy_with_logits(logits.float(), labels, pos_weight=POS_WEIGHT)

                loss_scaled = loss / GRAD_ACCUM

            scaler.scale(loss_scaled).backward()
            running_loss += float(loss.detach())

            if b_idx % GRAD_ACCUM == 0 or b_idx == num_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            # live heartbeat
            if b_idx == 1 or b_idx == 10 or b_idx % LOG_EVERY == 0 or b_idx == num_batches:
                elapsed_ep = time.time() - ep_t0
                speed = (b_idx * BATCH_SIZE) / max(elapsed_ep, 1e-3)
                eta_min = ((num_batches - b_idx) * (elapsed_ep / b_idx)) / 60.0
                curr_lr = scheduler.get_last_lr()[0]
                log(f"[{tag} | ep {epoch}/{epochs}] batch {b_idx:4d}/{num_batches} ({b_idx / num_batches:5.1%}) | "
                    f"loss {running_loss / b_idx:.4f} | lr {curr_lr:.2e} | {speed:.1f} cases/s | "
                    f"elapsed {elapsed_ep / 60:.1f}m | eta {eta_min:.1f}m")

        log(f"evaluating [{tag}] epoch {epoch} and calibrating thresholds on dev...")
        metrics = evaluate_dev(model, dev_loader, calibrate=True, verbose=False)
        elapsed_total = (time.time() - exp_t0) / 60.0
        print(f"  [{tag}] ep {epoch}: loss {running_loss / num_batches:.4f} | dev loss {metrics['val_loss']:.4f} | "
              f"macro F1 {metrics['macro_f1']:.4f} | micro F1 {metrics['micro_f1']:.4f} | "
              f"subset acc {metrics['subset_acc']:.4f} | {elapsed_total:.1f} min")

        if metrics["macro_f1"] > best_f1:
            best_f1 = metrics["macro_f1"]
            if save_best_to:
                torch.save({
                    "trainable_state_dict": trainable_state_dict(model),
                    "macro_f1": best_f1,
                    "micro_f1": metrics["micro_f1"],
                    "subset_acc": metrics["subset_acc"],
                    "thresholds": metrics["thresholds"],
                    "epoch": epoch,
                }, save_best_to)
                print(f"  new best, saved {Path(save_best_to).name} (macro F1 {best_f1:.4f})")

    return best_f1

## Train

In [ ]:
EXPERIMENT_RESULTS = {}
# the best dev run so far (lora, lorma or full) is the teacher for distillation
BEST_TEACHER_INFO = {"macro_f1": -1.0, "exp_num": None, "method": None, "ckpt_path": None}

for exp in EXPERIMENTS:
    num, name, method = exp["num"], exp["name"], exp["method"]
    key = f"Exp {num}: {name}"
    zip_file = WORK_DIR / f"submission_exp{num}_{name}.zip"
    ckpt_file = BASE_DIR / f"best_lsi_exp{num}_{name}.pt"
    print(f"\n===== exp {num}: {name} =====")

    # restored from an earlier session: no retraining, but it can still be the teacher
    if SKIP_IF_ZIP_EXISTS and zip_file.exists() and ckpt_file.exists() and not SMOKE:
        ck = torch.load(ckpt_file, map_location="cpu", weights_only=False)
        b_f1 = float(ck.get("macro_f1", 0.0))
        log(f"found {zip_file.name} and its checkpoint (macro F1 {b_f1:.4f}), skipping")
        if method != "distillation" and b_f1 > BEST_TEACHER_INFO["macro_f1"]:
            BEST_TEACHER_INFO = {"macro_f1": b_f1, "exp_num": num, "method": method, "ckpt_path": str(ckpt_file)}
        EXPERIMENT_RESULTS[key] = {
            "macro_f1": round(b_f1, 4),
            "micro_f1": round(float(ck.get("micro_f1", 0.0)), 4),
            "subset_acc": round(float(ck.get("subset_acc", 0.0)), 4),
            "zip_path": str(zip_file),
        }
        del ck
        continue

    seed_everything(SEED + num)

    teacher_map = None
    if exp.get("kd"):
        assert BEST_TEACHER_INFO["ckpt_path"] is not None, "no teacher checkpoint available for distillation"
        log(f"teacher: {BEST_TEACHER_INFO['method'].upper()} (dev macro F1 {BEST_TEACHER_INFO['macro_f1']:.4f})")
        teacher = make_model(BEST_TEACHER_INFO["method"])
        t_ck = torch.load(BEST_TEACHER_INFO["ckpt_path"], map_location="cpu", weights_only=False)
        teacher.load_state_dict(t_ck["trainable_state_dict"], strict=False)
        # the teacher logits are computed once over the train set and reused every epoch
        eval_train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=lsi_collate_fn)
        t_ids, t_lg, _ = get_split_logits(teacher, eval_train_loader, desc="Teacher Soft Logits")
        teacher_map = {cid: lg.astype(np.float16) for cid, lg in zip(t_ids, t_lg)}
        del teacher, t_ck
        gc.collect()
        if USE_AMP:
            torch.cuda.empty_cache()
        log(f"cached teacher logits for {len(teacher_map)} cases")

    model = make_model(method)
    log(f"starting exp {num}/{len(EXPERIMENTS)}: {name} (epochs {EPOCHS}, backbone lr {exp['backbone_lr']}, head lr {exp['head_lr']})")
    run_training(model, exp, EPOCHS, teacher_map=teacher_map, save_best_to=ckpt_file)

    # go back to the best dev epoch for the final dev report and the zip
    best_ck = torch.load(ckpt_file, map_location="cpu", weights_only=False)
    model.load_state_dict(best_ck["trainable_state_dict"], strict=False)
    best_thresholds = best_ck["thresholds"]
    final_dev = evaluate_dev(model, dev_loader, thresholds=best_thresholds, calibrate=False, verbose=True)
    best_f1 = final_dev["macro_f1"]

    if method != "distillation" and best_f1 > BEST_TEACHER_INFO["macro_f1"]:
        BEST_TEACHER_INFO = {"macro_f1": best_f1, "exp_num": num, "method": method, "ckpt_path": str(ckpt_file)}
        log(f"{name} is the best teacher so far (dev macro F1 {best_f1:.4f})")

    generate_submission(model, best_thresholds, num, name)

    EXPERIMENT_RESULTS[key] = {
        "macro_f1": round(best_f1, 4),
        "micro_f1": round(final_dev["micro_f1"], 4),
        "subset_acc": round(final_dev["subset_acc"], 4),
        "precision": round(final_dev["precision"], 4),
        "recall": round(final_dev["recall"], 4),
        "zip_path": str(zip_file),
    }

    del model, best_ck, teacher_map
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()

# summary of all experiments
print("\n" + "=" * 100)
print(f"{'Experiment':<22} | {'Macro-F1':<10} | {'Micro-F1':<10} | {'Subset Acc':<10} | zip")
print("-" * 100)
for exp_name, res in EXPERIMENT_RESULTS.items():
    print(f"{exp_name:<22} | {res['macro_f1']:<10.4f} | {res['micro_f1']:<10.4f} | {res['subset_acc']:<10.4f} | {res['zip_path']}")
print("=" * 100)

with open(WORK_DIR / "phase5_lsi_summary.json", "w", encoding="utf-8") as fh:
    json.dump(EXPERIMENT_RESULTS, fh, indent=2)